# Week 13 Lecture: Unsupervised Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-13-lecture.ipynb)

Every model so far learned from a target `y`. This week there is no `y`. You'll use K-Means to find groups of similar rows, turn those clusters into named segments, and use PCA to compress many correlated columns into a few components.

## How to use this notebook

This notebook accompanies the Week 13 Tuesday lecture. You can:

- **Follow along during class**: run each cell as the instructor discusses it
- **Work the Your Turn exercises** when prompted
- **Review after class**: rerun and experiment with the code

## Setup

Everything we need is already installed in Colab. Today uses two datasets:

- **Customers**: a small simulated dataset of shoppers, with each customer's annual spending and number of store visits per year. There is no label telling us which "type" of customer each one is.
- **Wine**: scikit-learn's built-in wine dataset, with 13 chemical measurements for each of 178 wines from three cultivars.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

In [ ]:
# Simulated customers: annual spending ($) and visits per year
rng = np.random.default_rng(42)
customers = pd.DataFrame({
    "annual_spending": np.concatenate([
        rng.normal(2000, 500, 30),
        rng.normal(5000, 800, 30),
        rng.normal(8500, 700, 30),
    ]),
    "visits_per_year": np.concatenate([
        rng.normal(5, 2, 30),
        rng.normal(15, 3, 30),
        rng.normal(35, 5, 30),
    ]),
})

print(customers.shape)
customers.head()

In [ ]:
wine = load_wine(as_frame=True)
X_wine = wine.data            # 13 chemical measurements
y_wine = wine.target          # cultivar (0, 1, 2), used only to color plots and score models

print(X_wine.shape)
X_wine.head()

---

## 1. K-Means Clustering

*This section follows the K-Means part of the lecture.*

### Scale, then fit

K-Means groups customers by **distance**. In raw units a difference of \$1 counts as much as a difference of 1 visit, so spending (thousands of dollars) would drown out visits (a few dozen). `StandardScaler` puts both columns in standard deviations first, the same fix we used for KNN in week 12.

In [ ]:
customers_scaled = StandardScaler().fit_transform(customers)

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(customers_scaled)

customers.assign(cluster=cluster_labels).round(1).sample(4, random_state=42)   # a random handful, from all three clusters

**Reading the code:**

- `fit_predict` finds the centroids **and** returns each row's cluster number. There's no `y` anywhere.
- `n_init=10` runs K-Means from 10 different starts and keeps the one with the lowest **inertia** (total squared distance from each customer to its centroid).
- `random_state=42` makes those starts repeatable.
- The cluster numbers are just names. Cluster 0 isn't "first" or "best".

### 🎯 Your Turn: Does K-Means Always Agree?

In class, Start B got stuck with a worse answer than Start A. Let's see how often that happens.

1. For each `random_state` from **0 to 9**, fit `KMeans(n_clusters=3, init='random', n_init=1, random_state=seed)` on `customers_scaled` and print its inertia.
2. How many different inertias did you get? Which seeds got stuck like Start B?
3. Repeat step 1 with `n_init=10`. What changed, and why?

*Hint: `init='random'` picks three random customers as the starting centroids, just like the slide. (scikit-learn's default, `'k-means++'`, spreads the starts out more carefully, so it gets stuck less often.) `.fit(customers_scaled).inertia_` gives the inertia.*

In [ ]:
# Your code here

### Choosing K

We picked K = 3 because we could *see* three groups. With more than two features you can't, so fit several values of K and score each one two ways:

- **Inertia** always drops as K grows. Look for the **elbow**, where adding a cluster stops buying much.
- **Silhouette score** (−1 to +1) compares how close each customer is to its own cluster vs. the next-nearest one. Look for the **peak**.

In [ ]:
k_values = range(2, 9)
inertias, silhouettes = [], []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(customers_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(customers_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(k_values, inertias, marker="o", color="#399de5")
ax1.set(title="Elbow: inertia by K", xlabel="K (number of clusters)", ylabel="Inertia (lower = tighter)")
ax2.plot(k_values, silhouettes, marker="o", color="#e58139")
ax2.set(title="Silhouette score by K", xlabel="K (number of clusters)", ylabel="Silhouette (higher = better)")
plt.tight_layout()
plt.show()

Both curves point to the same K here: a sharp elbow and the highest silhouette land on the same value. Real data is rarely this tidy, and when the two disagree, you'll also weigh what the business can actually use.

### From clusters to segments

K-Means hands back cluster numbers. **Profile** each cluster by averaging its features, **name** it in plain business terms, and it becomes a **segment** you can act on. We sort by spending first, because the cluster numbers themselves are arbitrary.

In [ ]:
segment_names = ["Occasional shoppers", "Regulars", "Loyal big spenders"]

(customers
 .assign(cluster=cluster_labels)
 .groupby("cluster")
 .agg(customers=("annual_spending", "size"),
      avg_spending=("annual_spending", "mean"),
      avg_visits=("visits_per_year", "mean"))
 .sort_values("avg_spending")
 .assign(segment=segment_names)   # names chosen AFTER reading the profile
 .round(1))

### 🎯 Your Turn: Skip the Scaler

The marketing team adds each customer's **household income**, measured in dollars. Run the cell below to build `customers_income`, then:

1. Fit `KMeans(n_clusters=3, n_init=10, random_state=42)` on `customers_income` **without** scaling, and profile the clusters (mean of each column by cluster).
2. Do the same **with** `StandardScaler` first.
3. Which version still finds the occasional shoppers, regulars and loyal big spenders? What did the unscaled clusters split on, and why?

*Hint: reuse the profiling chain above, with `.groupby("cluster").mean()` instead of the named `agg`. Then compare `customers_income.std()` across the three columns.*

In [ ]:
# Household income in dollars: a little higher for bigger spenders, but very spread out
income_rng = np.random.default_rng(7)
customers_income = customers.assign(
    household_income=np.concatenate([income_rng.normal(mean, 20000, 30) for mean in (55000, 65000, 75000)])
)
customers_income.round(1).head()

In [ ]:
# Your code here

---

## 2. PCA

*This section follows the PCA part of the lecture.*

### Scale first

PCA looks for the directions where the data **varies** the most. If one column is measured in much bigger numbers than the rest, its variance swamps everything else. Let's fit PCA on the wine data with and without scaling.

In [ ]:
pca_unscaled = PCA().fit(X_wine)

wine_scaled = StandardScaler().fit_transform(X_wine)
pca = PCA().fit(wine_scaled)

print(f"PC1 share of the variance, unscaled: {pca_unscaled.explained_variance_ratio_[0]:.1%}")
print(f"PC1 share of the variance, scaled:   {pca.explained_variance_ratio_[0]:.1%}")

In [ ]:
# Why? Compare each column's spread
X_wine.std().sort_values(ascending=False).round(1).head()

Unscaled, PC1 is basically `proline` alone, because proline's numbers are so much larger than every other column's. After `StandardScaler`, every column has a standard deviation of 1, so each gets a fair say. From here on, we use the scaled version.

### How many components?

`explained_variance_ratio_` gives each component's share of the total variance. A **scree plot** shows those shares as bars, and the running total shows how much you keep if you stop at each count.

In [ ]:
var_pct = 100 * pca.explained_variance_ratio_
cum_pct = var_pct.cumsum()
components = range(1, len(var_pct) + 1)

plt.figure(figsize=(9, 4))
plt.bar(components, var_pct, color="#e58139", label="Each component")
plt.plot(components, cum_pct, marker="o", color="#399de5", label="Running total")
plt.axhline(90, color="gray", linestyle="--", label="90%")
plt.xlabel("Principal component")
plt.ylabel("Variance explained (%)")
plt.xticks(list(components))
plt.legend()
plt.show()

print(f"First 2 components: {cum_pct[1]:.1f}% of the variance")
print(f"Components needed for 90%: {(cum_pct >= 90).argmax() + 1}")

### What does PC1 mean?

Each component is a weighted mix of the original columns. Those weights are its **loadings** (`pca.components_`). Big positive and big negative weights tell you which columns move together in that component.

In [ ]:
pc1_loadings = pd.Series(pca.components_[0], index=X_wine.columns).sort_values()

pc1_loadings.plot.barh(figsize=(7, 5), color=["#399de5" if v < 0 else "#e58139" for v in pc1_loadings])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("PC1 loading (weight)")
plt.show()

Flavanoids, total phenols and related measures all load strongly in the same direction, so PC1 roughly measures "phenolic richness". The sign itself is arbitrary: another run could flip every weight. What matters is which columns move together.

### 13 columns, one picture

Two components are enough to draw the wines on a flat plot. PCA never sees the cultivar labels; we add the colors afterwards only to check what PCA found.

In [ ]:
wine_2d = PCA(n_components=2).fit_transform(wine_scaled)

plt.figure(figsize=(7, 5))
for cultivar, color in zip(range(3), ["#e58139", "#399de5", "#59a14f"]):
    mask = (y_wine == cultivar).to_numpy()
    plt.scatter(wine_2d[mask, 0], wine_2d[mask, 1], color=color, label=f"Cultivar {cultivar}", edgecolor="white")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)")
plt.legend()
plt.show()

### PCA in a pipeline

PCA can also feed a model. Following week 12's habits, we split first, lock the test set away, and compare choices with 5-fold CV on the training set only. PCA goes **inside** the pipeline, so it's refit on each fold's training part and never sees that fold's validation rows.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.2, stratify=y_wine, random_state=42)

all_13 = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
pcs_2 = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("model", LogisticRegression(max_iter=1000)),
])

for name, pipe in [("All 13 columns", all_13), ("2 PCA components", pcs_2)]:
    print(f"{name:>16}: 5-fold CV accuracy {cross_val_score(pipe, X_train, y_train, cv=5).mean():.3f}")

### 🎯 Your Turn: How Many Components Do You Need?

Two components lose a little accuracy. How many do you need to get it back?

1. For each component count from **1 to 13**, build a `scale → PCA(n_components=k) → LogisticRegression(max_iter=1000)` pipeline and record its mean 5-fold CV accuracy on `X_train`, `y_train`.
2. Plot CV accuracy against the number of components, with a horizontal line at the all-13-columns accuracy.
3. What's the **smallest** count whose CV accuracy is within **one percentage point** (0.01) of the all-columns pipeline? Does accuracy always go up as you add components?

*Hint: copy `pcs_2` into a `for k in range(1, 14):` loop, changing `n_components=2` to `n_components=k`, and append each `cross_val_score(...).mean()` to a list. `plt.axhline(y)` draws the reference line.*

In [ ]:
# Your code here

---

## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `StandardScaler()` | Scale features to mean 0, std 1, so no feature dominates distances or variance |
| `KMeans(n_clusters=k, n_init=10, random_state=42)` | Group rows into k clusters, keeping the best of 10 random starts |
| `kmeans.fit_predict(X_scaled)` | Fit the clusters and return each row's cluster number |
| `kmeans.inertia_` | Total squared distance from rows to their centroids (the elbow method) |
| `silhouette_score(X_scaled, labels)` | How well separated the clusters are, from −1 to +1 (higher is better) |
| `df.assign(cluster=labels).groupby('cluster').mean()` | Profile clusters so you can name them as segments |
| `PCA(n_components=k)` | Combine correlated columns into k uncorrelated components |
| `pca.explained_variance_ratio_` | Each component's share of the total variance (the scree plot) |
| `pca.components_` | Loadings: each component's weight on every original column |
| `PCA(n_components=2).fit_transform(X_scaled)` | Squeeze many columns into 2 for a picture |
| `Pipeline([('scale', ...), ('pca', PCA(...)), ('model', ...)])` | Use PCA as a preprocessing step, refit inside every CV fold |